In [ ]:
# ORIGINAL SCRIPT


"""
Script to estimate the width of rights of way polygons
"""
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString, Point

# row = gpd.read_file(r'C:\Exclusion_Analysis_Phase3\CONUS_ROW_Width_Workspace\Revised_ROWs_CONUS_Exclusion_Analysis_Results_Area2Acres.gpkg')
# hw = gpd.read_file(r'C:\Exclusion_Analysis_Phase3\CONUS_ROW_Width_Workspace\NHS_CONUS.gpkg').to_crs(row.crs)


def get_row_width(hw_lines, row, initial_line_length=800, interval=100):
    """
    Generate perpendicular lines along highway centerlines, clip them to ROW polygons,
    save the geometries to GeoPackages, and compute average width.
    """

    # ---------------------------------------------------------
    # Step 1: Explode ROW geometry
    # ---------------------------------------------------------
    row = row.explode(index_parts=True).reset_index(drop=True)
    row["row_id"] = row.index

    # ---------------------------------------------------------
    # Step 2: Reproject highways to ROW CRS
    # ---------------------------------------------------------
    if hw_lines.crs != row.crs:
        hw_lines = hw_lines.to_crs(row.crs)

    # Containers for output
    perpendicular_lines_list = []
    clipped_perpendicular_records = []

    # ---------------------------------------------------------
    # Step 3: Generate perpendicular lines
    # ---------------------------------------------------------
    for _, hw in hw_lines.iterrows():
        line = hw.geometry

        for dist in np.arange(0, line.length, interval):

            # point on the line
            point = line.interpolate(dist)

            # tangent via micro-offsets
            tangent_start = line.interpolate(max(dist - 1e-5, 0))
            tangent_end = line.interpolate(min(dist + 1e-5, line.length))

            dx = tangent_end.x - tangent_start.x
            dy = tangent_end.y - tangent_start.y

            # Perpendicular direction
            perpendicular_dx = -dy
            perpendicular_dy = dx

            norm = np.sqrt(perpendicular_dx**2 + perpendicular_dy**2)
            if norm == 0:
                continue

            perpendicular_dx /= norm
            perpendicular_dy /= norm

            # Build perpendicular segment
            half_length = initial_line_length / 2
            p1 = Point(point.x - perpendicular_dx * half_length, point.y - perpendicular_dy * half_length)
            p2 = Point(point.x + perpendicular_dx * half_length, point.y + perpendicular_dy * half_length)

            perpendicular_line = LineString([p1, p2])

            # Store perpendicular line for output
            perpendicular_lines_list.append(perpendicular_line)

            # ---------------------------------------------------------
            # Step 4: Clip perpendicular line to ROW polygons
            # ---------------------------------------------------------
            for _, poly in row.iterrows():

                clipped = perpendicular_line.intersection(poly.geometry)

                if clipped.is_empty:
                    continue

                # Ensure list of individual line segments
                if clipped.geom_type == "LineString":
                    line_segments = [clipped]
                else:
                    line_segments = [
                        geom for geom in clipped.geoms
                        if geom.geom_type == "LineString"
                    ]

                for segment in line_segments:
                    clipped_perpendicular_records.append({
                        "row_id": poly["row_id"],
                        "length_m": segment.length,
                        "geometry": segment
                    })

    # ---------------------------------------------------------
    # Step 5: Save perpendicular lines to GeoPackage
    # ---------------------------------------------------------
    perpendicular_lines_gdf = gpd.GeoDataFrame(
        geometry=perpendicular_lines_list,
        crs=row.crs
    )
    perpendicular_lines_gdf.to_file("perpendicular_lines.gpkg", driver="GPKG")

    # ---------------------------------------------------------
    # Step 6: Save clipped perpendicular lines
    # ---------------------------------------------------------
    if len(clipped_perpendicular_records) == 0:
        row["approx_length_meters"] = 0
        return row

    clipped_perpendicular_gdf = gpd.GeoDataFrame(
        clipped_perpendicular_records,
        geometry="geometry",
        crs=row.crs
    )
    clipped_perpendicular_gdf.to_file("clipped_perpendicular_lines.gpkg", driver="GPKG")

    # ---------------------------------------------------------
    # Step 7: Compute average clipped perpendicular length per ROW polygon
    # ---------------------------------------------------------
    grouped = clipped_perpendicular_gdf.groupby("row_id")["length_m"].mean()
    row = row.copy()
    row["approx_length_meters"] = row["row_id"].map(grouped).fillna(0)

    row.to_file("row_with_widths.gpkg", driver="GPKG")

    return row

get_row_width(hw_lines=hw, row=row, initial_line_length=1200, interval=100)


In [1]:
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Point
import pandas as pd
import os
import logging
import time
from datetime import timedelta

# =====================================================
# CONFIGURATION
# =====================================================
ROOT_DIR = r"C:\Users\KyleSteen\Documents\ROW_Width_Analysis\CONUS"

ROW_GPKG = os.path.join(
    ROOT_DIR,
    "CONUS_Erase_NHS_TIGER_CZ_RP_neg5mBuffer_pos5mBuffer_multi_200m.gpkg"
)
HW_GPKG = os.path.join(ROOT_DIR, "NHS_CONUS.gpkg")

PERP_LINES_GPKG = os.path.join(ROOT_DIR, "CONUS_perpendicular_lines.gpkg")
CLIPPED_LINES_GPKG = os.path.join(ROOT_DIR, "CONUS_clipped_perpendicular_lines.gpkg")
CSV_OUTPUT = os.path.join(ROOT_DIR, "CONUS_row_widths.csv")

INITIAL_LINE_LENGTH = 2000   # meters
INTERVAL = 10                # meters

# =====================================================
# LOGGER
# =====================================================
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()

def eta(start, done, total):
    if done == 0:
        return "estimating..."
    rate = done / (time.time() - start)
    return str(timedelta(seconds=int((total - done) / rate)))

start_time = time.time()
logger.info("Starting ROW width workflow (optimized)...")

# =====================================================
# LOAD DATA
# =====================================================
logger.info("Loading ROW polygons...")
rows = gpd.read_file(ROW_GPKG)

for col in ["ROW_ID", "Square_Meters"]:
    if col not in rows.columns:
        raise KeyError(f"Missing required column: {col}")

logger.info(f"Loaded {len(rows):,} ROW polygons")

logger.info("Loading highway centerlines...")
hw_lines = gpd.read_file(HW_GPKG)

if hw_lines.crs != rows.crs:
    hw_lines = hw_lines.to_crs(rows.crs)

logger.info(f"Loaded {len(hw_lines):,} highways")

# =====================================================
# GENERATE PERPENDICULAR TRANSECTS
# =====================================================
logger.info("Generating perpendicular transects...")
perpendicular_lines = []

hw_start = time.time()
total_hw = len(hw_lines)

for i, hw_row in hw_lines.iterrows():
    line = hw_row.geometry

    for dist in np.arange(0, line.length, INTERVAL):
        point = line.interpolate(dist)

        tangent_start = line.interpolate(max(dist - 1e-5, 0))
        tangent_end = line.interpolate(min(dist + 1e-5, line.length))

        dx = tangent_end.x - tangent_start.x
        dy = tangent_end.y - tangent_start.y

        perp_dx = -dy
        perp_dy = dx
        norm = np.hypot(perp_dx, perp_dy)

        if norm == 0:
            continue

        perp_dx /= norm
        perp_dy /= norm

        half = INITIAL_LINE_LENGTH / 2

        p1 = Point(point.x - perp_dx * half, point.y - perp_dy * half)
        p2 = Point(point.x + perp_dx * half, point.y + perp_dy * half)

        perpendicular_lines.append(LineString([p1, p2]))

    if (i + 1) % 100 == 0:
        logger.info(
            f"Highways {i+1:,}/{total_hw:,} "
            f"({(i+1)/total_hw:.1%}) | ETA {eta(hw_start, i+1, total_hw)}"
        )

logger.info(f"Generated {len(perpendicular_lines):,} perpendicular lines")

perp_gdf = gpd.GeoDataFrame(geometry=perpendicular_lines, crs=rows.crs)

logger.info(f"Writing perpendicular lines → {PERP_LINES_GPKG}")
perp_gdf.to_file(PERP_LINES_GPKG, driver="GPKG")

# =====================================================
# SPATIAL INDEX PRE-FILTER (MAJOR SPEED BOOST)
# =====================================================
logger.info("Running spatial join (bbox prefilter)...")

sjoin_start = time.time()

# Spatial join uses R-tree index automatically
candidate_pairs = gpd.sjoin(
    perp_gdf,
    rows[["ROW_ID", "geometry"]],
    how="inner",
    predicate="intersects"
)

logger.info(
    f"Spatial join produced {len(candidate_pairs):,} candidate pairs "
    f"in {timedelta(seconds=int(time.time() - sjoin_start))}"
)

# =====================================================
# PRECISE INTERSECTION (FASTER THAN FULL OVERLAY)
# =====================================================
logger.info("Computing precise intersections...")

clip_start = time.time()

candidate_pairs["geometry"] = candidate_pairs.apply(
    lambda row: row.geometry.intersection(
        rows.loc[row.index_right, "geometry"]
    ),
    axis=1
)

clipped_gdf = gpd.GeoDataFrame(
    candidate_pairs[["ROW_ID", "geometry"]],
    crs=rows.crs
)

logger.info(
    f"Computed intersections in "
    f"{timedelta(seconds=int(time.time() - clip_start))}"
)

logger.info(f"Writing clipped lines → {CLIPPED_LINES_GPKG}")
clipped_gdf.to_file(CLIPPED_LINES_GPKG, driver="GPKG")

# =====================================================
# WIDTH AGGREGATION → CSV
# =====================================================
logger.info("Computing average ROW widths...")

clipped_gdf["length_m"] = clipped_gdf.geometry.length

avg_widths = (
    clipped_gdf
    .groupby("ROW_ID")["length_m"]
    .mean()
    .reset_index()
    .rename(columns={"length_m": "Approximate_Width_Meters"})
)

output = rows[["ROW_ID", "Square_Meters"]].merge(
    avg_widths,
    on="ROW_ID",
    how="left"
)

logger.info(f"Writing CSV → {CSV_OUTPUT}")
output.to_csv(CSV_OUTPUT, index=False)

logger.info(
    f"Workflow complete in {timedelta(seconds=int(time.time() - start_time))}"
)

[2026-02-28 16:20:41] Starting ROW width workflow (optimized)...
[2026-02-28 16:20:41] Loading ROW polygons...
[2026-02-28 16:20:54] Loaded 850,941 ROW polygons
[2026-02-28 16:20:54] Loading highway centerlines...
C:\Users\KyleSteen\miniconda3\envs\jupyter311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column FUT_YEAR, 2038-12-31T00:00:00.0Z, successfully parsed
  return ogr_read(
[2026-02-28 16:21:05] Loaded 480,294 highways
[2026-02-28 16:21:05] Generating perpendicular transects...
[2026-02-28 16:21:07] Highways 100/480,294 (0.0%) | ETA 2:16:41
[2026-02-28 16:21:07] Highways 200/480,294 (0.0%) | ETA 1:31:37
[2026-02-28 16:21:08] Highways 300/480,294 (0.1%) | ETA 1:28:43
[2026-02-28 16:21:09] Highways 400/480,294 (0.1%) | ETA 1:18:19
[2026-02-28 16:21:10] Highways 500/480,294 (0.1%) | ETA 1:19:08
[2026-02-28 16:21:11] Highways 600/480,294 (0.1%) | ETA 1:21:28
[2026-02-28 16:21:12] Highways 700/480,294 (0.1%) | ETA 1:24:07
[2026-02-28 1